## Suppliers Table - PySpark Cleaning
This table does not need transformations. We validate the data and fix the date column type.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, to_date

spark = SparkSession.builder.appName("SuppliersCleaning").getOrCreate()

df_suppliers = spark.read.csv("../Coursework_dataset/suppliers.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 18:28:23 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 18:28:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:28:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:28:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 18:28:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/14 18:28:24 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/14 18:28:24 WARN Utils: Servic

### Preview the suppliers table

In [2]:
df_suppliers.show(5)

+-----------+--------------------+-------+--------------------+--------------+
|supplier_id|       supplier_name|country|       contact_email|contract_start|
+-----------+--------------------+-------+--------------------+--------------+
|   supp_001|Himalayan Traders...|  Nepal|contact@himalayan...|    2016-04-01|
|   supp_002|Sagarmatha Suppliers|  Nepal|contact@sagarmath...|    2015-04-13|
|   supp_003|  Nepal Commerce Hub|  Nepal|contact@nepalcomm...|    2018-01-31|
|   supp_004|Kathmandu Wholesa...|  Nepal|contact@kathmandu...|    2017-09-30|
|   supp_005|Lumbini Trading H...|  Nepal|contact@lumbinitr...|    2017-07-03|
+-----------+--------------------+-------+--------------------+--------------+
only showing top 5 rows


### Check how many values each column has (non-null count)

In [3]:
non_null_counts = df_suppliers.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_suppliers.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,supplier_id,200
1,supplier_name,200
2,country,200
3,contact_email,200
4,contract_start,200


### Check which columns have null values

In [4]:
null_counts = df_suppliers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_suppliers.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,supplier_id,0
1,supplier_name,0
2,country,0
3,contact_email,0
4,contract_start,0


### Check for duplicate supplier_id rows

In [5]:
duplicates = df_suppliers.join(
    df_suppliers.groupBy("supplier_id").count().filter(col("count") > 1).select("supplier_id"),
    on="supplier_id",
    how="inner"
)

print("Duplicate supplier_id rows:", duplicates.count())

Duplicate supplier_id rows: 0


### Fix: Convert contract_start to a proper date type
Stored as a string - we cast it to DateType for correct date handling.

In [6]:
df_suppliers = df_suppliers.withColumn("contract_start", to_date(col("contract_start")))

print("Unparseable contract_start values:", df_suppliers.filter(col("contract_start").isNull()).count())

Unparseable contract_start values: 0


### Check the schema to confirm data types

In [7]:
df_suppliers.printSchema()

root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- contact_email: string (nullable = true)
 |-- contract_start: date (nullable = true)



### Export the cleaned suppliers table

In [8]:
df_suppliers.coalesce(1).write.csv("../cleaned_dataset/suppliers_cleaned.csv", header=True, mode="overwrite")
print("Suppliers table exported successfully to 'cleaned_dataset/suppliers_cleaned.csv'")

Suppliers table exported successfully to 'cleaned_dataset/suppliers_cleaned.csv'
